# K-Means Clustering

Implement Lloyd's k-means algorithm from scratch (k-means++ init, tensorised distance computation, convergence check), validate against scikit-learn, and explore cluster quality on synthetic blobs.

## Configuration

Device, seed, and dtype come from `config.toml` via `shared.config.configure()`.

In [1]:
import sys
from pathlib import Path

import matplotlib

matplotlib.use("Agg")  # headless-safe under nbconvert
import matplotlib.pyplot as plt  # noqa: E402
import torch  # noqa: E402


def _find_repo_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "pyproject.toml").exists():
            return p
    return start


REPO_ROOT = _find_repo_root(Path.cwd())
sys.path.insert(0, str(REPO_ROOT))

from shared.config import configure  # noqa: E402

device = configure()
print("running on:", device)

running on: mps


## Lloyd's algorithm

K-means minimises the **within-cluster sum of squares** (inertia):

$$J = \sum_{i=1}^{n} \|x_i - \mu_{c_i}\|_2^2$$

where \(\mu_k\) is the centroid of cluster \(k\) and \(c_i\) is the assignment for point \(i\).

Lloyd's algorithm alternates:

1. **Assignment step**: \(c_i = \arg\min_k \|x_i - \mu_k\|_2^2\)
2. **Update step**: \(\mu_k = \frac{1}{|C_k|} \sum_{i \in C_k} x_i\)

Each iteration can only decrease \(J\), so the algorithm always converges (in finitely many steps to a local minimum).

### K-means++ initialisation

Random initialisation can land in poor local optima.  K-means++ spreads initial centroids by sampling each successive centroid proportional to the squared distance to the nearest already-chosen centroid.  This costs \(O(k \cdot n)\) extra at startup but often finds much better solutions.

In [2]:
import numpy as np
from sklearn.datasets import make_blobs

# Reproducible synthetic blobs with 4 true clusters
X_np, y_true_np = make_blobs(
    n_samples=600,
    centers=4,
    cluster_std=1.8,  # larger std makes clusters overlap so init sensitivity is visible
    random_state=42,
)

X = torch.tensor(X_np, dtype=torch.float32, device=device)  # (600, 2)

print(f"Dataset: {X.shape}  dtype={X.dtype}  device={X.device}")
print(f"True cluster counts: {np.bincount(y_true_np)}")

# Quick scatter of true labels
fig, ax = plt.subplots(figsize=(6, 5))
for k in range(4):
    m = y_true_np == k
    ax.scatter(X_np[m, 0], X_np[m, 1], s=18, alpha=0.6, label=f"cluster {k}")
ax.set_title("Ground-truth clusters (not seen by k-means)")
ax.legend()
fig.tight_layout()
plt.savefig("kmeans_gt.png", dpi=80)
plt.show()

Dataset: torch.Size([600, 2])  dtype=torch.float32  device=mps:0
True cluster counts: [150 150 150 150]


/var/folders/gx/cg22rrrs5mx_t0gwgx3809t80000gn/T/ipykernel_75443/2277169183.py:26: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## K-Means from scratch

The implementation uses:
- **Tensorised squared distances** via broadcasting (\(\|x - \mu\|^2 = \|x\|^2 - 2x^\top\mu + \|\mu\|^2\)) for efficiency.
- **K-means++ initialisation** for better starting centroids.
- **Empty-cluster handling**: if an update step produces an empty cluster (rare on clean data), that centroid is re-initialised to a random data point.
- **Convergence check**: stop when assignments do not change between iterations.

In [3]:
def kmeans_pp_init(X: torch.Tensor, k: int, generator: torch.Generator | None = None) -> torch.Tensor:
    """K-means++ initialisation: returns (k, d) tensor of initial centroids."""
    n, d = X.shape
    centroids: list[torch.Tensor] = []

    # Pick first centroid uniformly at random
    idx = torch.randint(n, (1,), generator=generator, device=X.device).item()
    centroids.append(X[idx])

    for _ in range(1, k):
        # Squared distances to the nearest centroid so far
        dists = torch.stack(
            [torch.sum((X - c) ** 2, dim=1) for c in centroids], dim=1
        ).min(dim=1).values  # (n,)

        # Sample proportional to squared distance (clamp to avoid negative values from float cancellation)
        dists = dists.clamp(min=0.0)
        probs = dists / dists.sum()
        chosen = torch.multinomial(probs, 1, generator=generator).item()
        centroids.append(X[chosen])

    return torch.stack(centroids, dim=0)  # (k, d)


def _sq_dists(X: torch.Tensor, C: torch.Tensor) -> torch.Tensor:
    """Compute (n, k) matrix of squared Euclidean distances via broadcasting.

    Uses identity ||x - c||^2 = ||x||^2 - 2 x^T c + ||c||^2 for O(nkd) work.
    """
    # X: (n, d), C: (k, d)
    xx = (X ** 2).sum(dim=1, keepdim=True)   # (n, 1)
    cc = (C ** 2).sum(dim=1, keepdim=True).T  # (1, k)
    xc = X @ C.T                              # (n, k)
    return xx + cc - 2 * xc                   # (n, k)


class KMeansScratch:
    """Lloyd's k-means with k-means++ init, run on a torch tensor."""

    def __init__(self, k: int, max_iters: int = 300, tol: float = 0.0, seed: int = 0):
        self.k = k
        self.max_iters = max_iters
        self.tol = tol
        self.seed = seed
        self.centroids_: torch.Tensor | None = None
        self.labels_: torch.Tensor | None = None
        self.inertia_: float = float("inf")
        self.n_iters_: int = 0

    def fit(self, X: torch.Tensor, init_centroids: "torch.Tensor | None" = None) -> "KMeansScratch":
        n, d = X.shape
        gen = torch.Generator(device=X.device)
        gen.manual_seed(self.seed)

        # Initialise centroids: use provided init_centroids if given, else k-means++
        if init_centroids is not None:
            C = init_centroids.clone()
        else:
            C = kmeans_pp_init(X, self.k, generator=gen)  # (k, d)

        labels_prev = torch.full((n,), -1, device=X.device, dtype=torch.long)

        for it in range(self.max_iters):
            # Assignment step
            dists = _sq_dists(X, C)        # (n, k)
            labels = dists.argmin(dim=1)   # (n,)

            # Convergence check (assignments unchanged)
            if torch.equal(labels, labels_prev):
                self.n_iters_ = it
                break
            labels_prev = labels.clone()

            # Update step — recompute centroids
            new_C = torch.zeros_like(C)
            for kk in range(self.k):
                members = X[labels == kk]
                if members.shape[0] == 0:
                    # Empty cluster: re-initialise to a random point
                    idx = torch.randint(n, (1,), generator=gen, device=X.device).item()
                    new_C[kk] = X[idx]
                else:
                    new_C[kk] = members.mean(dim=0)
            C = new_C
        else:
            self.n_iters_ = self.max_iters

        # Compute inertia
        final_dists = _sq_dists(X, C)
        self.inertia_ = final_dists[torch.arange(n), labels].sum().item()
        self.centroids_ = C
        self.labels_ = labels
        return self


K_TRUE = 4
km_scratch = KMeansScratch(k=K_TRUE, max_iters=300, seed=42)
km_scratch.fit(X)

print(f"From-scratch K-Means: inertia={km_scratch.inertia_:.2f}, iters={km_scratch.n_iters_}")
print(f"Cluster sizes: {[(km_scratch.labels_ == k).sum().item() for k in range(K_TRUE)]}")

From-scratch K-Means: inertia=3689.85, iters=8
Cluster sizes: [150, 152, 146, 152]


## Sklearn baseline

`sklearn.cluster.KMeans` with `init='k-means++'` is the standard reference implementation.

In [4]:
from sklearn.cluster import KMeans

sk_km = KMeans(n_clusters=K_TRUE, init="k-means++", n_init=10, random_state=42)
sk_km.fit(X_np)

print(f"sklearn KMeans: inertia={sk_km.inertia_:.2f}, iters={sk_km.n_iter_}")
print(f"Cluster sizes: {np.bincount(sk_km.labels_).tolist()}")

sklearn KMeans: inertia=3689.78, iters=19
Cluster sizes: [153, 150, 152, 145]


## Validation

We assert two things:

1. **Inertia check**: our inertia should be within 10% of sklearn's (which runs 10 restarts and picks the best).
2. **Assignment agreement**: ARI (Adjusted Rand Index) between our assignments and sklearn's should be ≥ 0.95, meaning the cluster shapes match up to label permutation.

In [5]:
from sklearn.metrics import adjusted_rand_score

our_inertia = km_scratch.inertia_
sk_inertia  = sk_km.inertia_

our_labels_np = km_scratch.labels_.cpu().numpy()
sk_labels_np  = sk_km.labels_

ari = adjusted_rand_score(sk_labels_np, our_labels_np)

print(f"Inertia — scratch: {our_inertia:.2f}  |  sklearn: {sk_inertia:.2f}")
print(f"Adjusted Rand Index (scratch vs sklearn): {ari:.4f}")

# Inertia should be within 10% of sklearn (sklearn uses 10 restarts)
assert our_inertia <= sk_inertia * 1.10, (
    f"Inertia too high: scratch={our_inertia:.2f} > sklearn*1.10={sk_inertia*1.10:.2f}"
)
# Cluster structure should match closely
assert ari >= 0.95, f"ARI too low: {ari:.4f} < 0.95"

print("PASS: inertia within 10% of sklearn and ARI ≥ 0.95")

Inertia — scratch: 3689.85  |  sklearn: 3689.78
Adjusted Rand Index (scratch vs sklearn): 0.9956
PASS: inertia within 10% of sklearn and ARI ≥ 0.95


## Cluster visualisation

We plot the discovered clusters and centroids for both implementations.

In [6]:
def plot_clusters(ax, X_np: np.ndarray, labels: np.ndarray,
                  centroids: np.ndarray, title: str) -> None:
    """Scatter plot of assigned clusters with centroid markers."""
    k = centroids.shape[0]
    colors = plt.cm.tab10.colors
    for kk in range(k):
        m = labels == kk
        ax.scatter(X_np[m, 0], X_np[m, 1], s=18, alpha=0.55,
                   color=colors[kk % len(colors)], label=f"cluster {kk}")
    ax.scatter(centroids[:, 0], centroids[:, 1],
               s=200, c="black", marker="X", zorder=5, label="centroids")
    ax.set_title(title)
    ax.legend(fontsize=7)


fig, axes = plt.subplots(1, 2, figsize=(12, 5))

plot_clusters(axes[0], X_np,
              km_scratch.labels_.cpu().numpy(),
              km_scratch.centroids_.cpu().numpy(),
              f"From-scratch K-Means  (inertia={km_scratch.inertia_:.1f})")

plot_clusters(axes[1], X_np,
              sk_km.labels_,
              sk_km.cluster_centers_,
              f"sklearn KMeans  (inertia={sk_km.inertia_:.1f})")

fig.tight_layout()
plt.savefig("kmeans_clusters.png", dpi=80)
plt.show()

/var/folders/gx/cg22rrrs5mx_t0gwgx3809t80000gn/T/ipykernel_75443/2625625975.py:30: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Elbow plot: choosing K

Running k-means for a range of \(K\) values reveals an "elbow" where adding more clusters yields diminishing inertia reduction.  The elbow suggests \(K=4\) here, but it is a heuristic — not proof that exactly 4 real clusters exist.

In [7]:
k_values = list(range(1, 10))
inertias_scratch = []
inertias_sklearn = []

for k in k_values:
    m_s = KMeansScratch(k=k, max_iters=300, seed=42)
    m_s.fit(X)
    inertias_scratch.append(m_s.inertia_)

    m_sk = KMeans(n_clusters=k, init="k-means++", n_init=10, random_state=42)
    m_sk.fit(X_np)
    inertias_sklearn.append(m_sk.inertia_)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(k_values, inertias_scratch, "o-", label="from-scratch", lw=2)
ax.plot(k_values, inertias_sklearn, "s--", label="sklearn", lw=2)
ax.axvline(K_TRUE, color="gray", linestyle=":", label=f"true K={K_TRUE}")
ax.set_xlabel("K")
ax.set_ylabel("Inertia (WCSS)")
ax.set_title("Elbow plot: inertia vs number of clusters")
ax.legend()
fig.tight_layout()
plt.savefig("kmeans_elbow.png", dpi=80)
plt.show()
print("Elbow plot saved.")

Elbow plot saved.


/var/folders/gx/cg22rrrs5mx_t0gwgx3809t80000gn/T/ipykernel_75443/3716611605.py:24: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Sensitivity to initialisation

K-means converges to a **local minimum** that depends on the initial centroids.  K-means++ reduces (but does not eliminate) bad starts.  Running multiple restarts and keeping the best (lowest inertia) is standard practice.

In [8]:
torch.manual_seed(0)
n_runs = 10
inertias_random  = []
inertias_pp      = []

for run in range(n_runs):
    # Random init: sample K_TRUE distinct points uniformly (different each run)
    rng = torch.Generator(device="cpu")
    rng.manual_seed(run * 9999 + 7)
    idx = torch.randperm(X.shape[0], generator=rng)[:K_TRUE]  # generated on cpu, used as index
    C_rand = X[idx].clone()  # (K_TRUE, d)

    m_rand = KMeansScratch(k=K_TRUE, max_iters=300, seed=run)
    m_rand.fit(X, init_centroids=C_rand)  # pass random starting centroids
    inertias_random.append(m_rand.inertia_)

    # K-means++ init: use built-in initialisation (no init_centroids)
    m_pp = KMeansScratch(k=K_TRUE, max_iters=300, seed=run)
    m_pp.fit(X)
    inertias_pp.append(m_pp.inertia_)

print("Inertia over 10 runs — random uniform init:")
print(f"  mean={np.mean(inertias_random):.2f}  std={np.std(inertias_random):.2f}  min={min(inertias_random):.2f}  max={max(inertias_random):.2f}")
print("\nInertia over 10 runs — k-means++ init:")
print(f"  mean={np.mean(inertias_pp):.2f}  std={np.std(inertias_pp):.2f}  min={min(inertias_pp):.2f}  max={max(inertias_pp):.2f}")
print(f"\nsklearn inertia (best of 10 restarts): {sk_km.inertia_:.2f}")

# Verify the demo shows real variance for random init
assert np.std(inertias_random) > 0, "Random init should show nonzero std across runs"
# K-means++ should achieve at least as good a best-case result as random init
assert min(inertias_pp) <= min(inertias_random) * 1.05, (
    f"K-means++ best ({min(inertias_pp):.2f}) should be <= random best ({min(inertias_random):.2f}) * 1.05"
)
print("\nPASS: random init shows nonzero variance; k-means++ is at least as good.")


Inertia over 10 runs — random uniform init:
  mean=4704.18  std=1242.53  min=3689.78  max=6286.24

Inertia over 10 runs — k-means++ init:
  mean=4201.56  std=1023.59  min=3689.78  max=6286.24

sklearn inertia (best of 10 restarts): 3689.78

PASS: random init shows nonzero variance; k-means++ is at least as good.


## Idiomatic sklearn usage

`KMeans` accepts `n_init='auto'` (sklearn ≥ 1.4) to automatically run multiple restarts.  The interface is consistent with other sklearn estimators: `fit`, `predict`, `fit_predict`, `transform`.

In [9]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# Pipeline: scale first, then cluster
pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("kmeans", KMeans(n_clusters=K_TRUE, init="k-means++", n_init=10, random_state=42)),
])
pipe.fit(X_np)
labels_pipe = pipe.named_steps["kmeans"].labels_
print(f"Pipeline KMeans inertia: {pipe.named_steps['kmeans'].inertia_:.2f}")
print(f"ARI (pipeline vs ground truth): {adjusted_rand_score(y_true_np, labels_pipe):.4f}")

Pipeline KMeans inertia: 107.44
ARI (pipeline vs ground truth): 0.9608


## Takeaways

- **Lloyd's algorithm** alternates assignment (nearest centroid) and update (cluster mean) steps; inertia decreases monotonically, guaranteeing convergence to a local minimum.
- **K-means++ initialisation** dramatically reduces the chance of poor local optima by spreading initial centroids; it is now the default in sklearn.
- **Inertia** (WCSS) measures compactness but always decreases with larger \(K\) — use elbow plots, silhouette score, or domain knowledge to choose \(K\).
- **Feature scaling** is critical: a feature with a large numerical range dominates Euclidean distance and can corrupt cluster assignments.
- K-means assumes **roughly spherical, similarly-sized clusters** — it can fail on elongated, non-convex, or varying-density clusters (consider DBSCAN or GMMs for those).
- In practice: use `sklearn.cluster.KMeans` with `n_init ≥ 10`; scale features with `StandardScaler`; validate with ARI (if labels are available) or silhouette/BIC.